# Task 1 - Netflix Content Recommendation System
### **Workflow Steps:**
1. **Step 1:** Prepare content-related features (Data cleaning & feature engineering).
2. **Step 2:** Convert text data into machine-readable format (TF-IDF Vectorization).
3. **Step 3:** Calculate content similarity scores (Cosine Similarity Matrix).
4. **Step 4:** Generate recommendations for selected titles (`get_recommendations`).
5. **Step 5:** Evaluate recommendation quality (Jaccard Genre Similarity, Type Consistency, Mean Similarity).


## **Environment Setup & Library Imports**

In [11]:
# Import essential libraries for data manipulation, NLP, and similarity computation
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Google Colab specific file upload check (optional)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Environment ready. In Colab: {IN_COLAB}")

Environment ready. In Colab: True


## **Step 1: Prepare Content-Related Features**

In this step, we:
1. Load `Dataset.csv` containing Netflix titles.
2. Inspect the dataset structure and missing values.
3. Clean placeholders such as `'Not Given'` and `NaN`.
4. Tokenize multi-word attributes (e.g., transforming director `"Steven Spielberg"` into `"stevenspielberg"` to prevent accidental overlap with other "Steven" or "Spielberg" names).
5. Combine all key metadata attributes (`type`, `listed_in` genres, `director`, `country`, `rating`) into a single unified text column called **`content_features`** (Metadata Soup).

In [2]:
# Load the dataset
dataset_path = '/content/Dataset.csv'

try:
    df = pd.read_csv(dataset_path)
except FileNotFoundError:
    print("Dataset.csv not found in current directory. Please upload Dataset.csv to Colab files.")
    if IN_COLAB:
        uploaded = files.upload()
        dataset_path = list(uploaded.keys())[0]
        df = pd.read_csv(dataset_path)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
display(df.head(3))

Dataset Shape: 8790 rows, 10 columns



,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"


In [3]:
# Clean missing values and 'Not Given' placeholders
target_columns = ['director', 'country', 'rating', 'listed_in', 'type']
for col in target_columns:
    df[col] = df[col].fillna('').astype(str).str.strip()
    df[col] = df[col].replace({'Not Given': '', 'nan': ''})

# Helper functions to sanitize metadata tokens
def sanitize_tokens(text):
    """Converts comma-separated multi-word names into single unified lowercase tokens."""
    if not text:
        return ''
    items = [item.strip().lower() for item in text.split(',') if item.strip()]
    tokens = [re.sub(r'[^a-zA-Z0-9]', '', item) for item in items]
    return ' '.join(tokens)

def clean_genres(text):
    """Cleans genre names and boosts their presence in the feature soup."""
    if not text:
        return ''
    genres = [g.strip().lower() for g in text.split(',') if g.strip()]
    clean_g = [re.sub(r'[^a-zA-Z0-9]', '', g) for g in genres]
    # Repeating genre tokens gives higher importance weight to genre alignment
    return ' '.join(clean_g + clean_g)

# Create consolidated feature soup
cleaned_genres = df['listed_in'].apply(clean_genres)
cleaned_directors = df['director'].apply(sanitize_tokens)
cleaned_countries = df['country'].apply(sanitize_tokens)
cleaned_rating = df['rating'].apply(lambda r: re.sub(r'[^a-zA-Z0-9]', '', r.lower()))
cleaned_type = df['type'].apply(lambda t: t.strip().lower())

df['content_features'] = (
    cleaned_type + ' ' +
    cleaned_genres + ' ' +
    cleaned_directors + ' ' +
    cleaned_countries + ' ' +
    cleaned_rating
)

# Create title lookup index (case-insensitive)
df['clean_title'] = df['title'].astype(str).str.strip()
indices = pd.Series(df.index, index=df['clean_title'].str.lower()).drop_duplicates()

print("Sample Feature Soup for First 3 Titles:")
for i in range(3):
    print(f"- {df.iloc[i]['clean_title']}: '{df.iloc[i]['content_features']}'")

Sample Feature Soup for First 3 Titles:
- Dick Johnson Is Dead: 'movie documentaries documentaries kirstenjohnson unitedstates pg13'
- Ganglands: 'tv show crimetvshows internationaltvshows tvactionadventure crimetvshows internationaltvshows tvactionadventure julienleclercq france tvma'
- Midnight Mass: 'tv show tvdramas tvhorror tvmysteries tvdramas tvhorror tvmysteries mikeflanagan unitedstates tvma'


## **Step 2: Convert Text Data into Machine-Readable Format (TF-IDF)**

Machines cannot directly compute similarities on raw text strings. We use **Term Frequency - Inverse Document Frequency (TF-IDF)**:
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

- **TF (Term Frequency):** Measures how frequently a tag appears in a title's feature soup.
- **IDF (Inverse Document Frequency):** Reduces the weight of ubiquitous terms (e.g., standard words) and increases the weight of distinctive genres or director names.
- The result is a high-dimensional vector space where every title is a point.

In [4]:
# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    stop_words='english',
    min_df=1,
    sublinear_tf=True
)

# Transform text features into a sparse TF-IDF matrix
tfidf_matrix = tfidf.fit_transform(df['content_features'])

print(f"TF-IDF Matrix Shape : {tfidf_matrix.shape[0]} titles x {tfidf_matrix.shape[1]} vocabulary features")
print(f"Vocabulary Sample   : {list(tfidf.vocabulary_.keys())[:15]}")

TF-IDF Matrix Shape : 8790 titles x 5126 vocabulary features
Vocabulary Sample   : ['movie', 'documentaries', 'kirstenjohnson', 'unitedstates', 'pg13', 'tv', 'crimetvshows', 'internationaltvshows', 'tvactionadventure', 'julienleclercq', 'france', 'tvma', 'tvdramas', 'tvhorror', 'tvmysteries']


## **Step 3: Calculate Content Similarity Scores (Cosine Similarity)**

We calculate the pairwise **Cosine Similarity** between all title vectors:
$$\text{Cosine Similarity}(\vec{u}, \vec{v}) = \frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\| \|\vec{v}\|}$$

- Scores range from `0.0` (completely dissimilar) to `1.0` (identical content attributes).
- The result is an $N \times N$ matrix where entry `[i, j]` is the similarity score between title $i$ and title $j$.

In [5]:
# Compute pairwise cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Similarity Matrix Shape : {cosine_sim.shape}")
print(f"Mean Pairwise Similarity: {np.mean(cosine_sim):.4f}")

Similarity Matrix Shape : (8790, 8790)
Mean Pairwise Similarity: 0.0634


## **Step 4: Generate Recommendations for Selected Titles**

The recommendation function `get_recommendations()`:
1. Accepts any title name (case-insensitive with partial/fuzzy fallback).
2. Retrieves the row index for the requested title.
3. Extracts pairwise similarity scores against all other $8,790+$ titles.
4. Sorts scores descending, excludes the title itself, and returns the top-$N$ most similar titles.

In [6]:
def get_recommendations(title: str, top_n: int = 5) -> pd.DataFrame:
    """
    Recommends top_n similar Netflix titles based on content features.
    """
    search_key = title.strip().lower()

    # Check exact match
    if search_key not in indices:
        # Check partial/fuzzy match
        matches = df[df['clean_title'].str.lower().str.contains(search_key, na=False)]
        if matches.empty:
            print(f"Title '{title}' not found in the dataset.")
            return pd.DataFrame()
        matched_title = matches.iloc[0]['clean_title']
        print(f"Exact match not found for '{title}'. Using closest match: '{matched_title}'")
        search_key = matched_title.lower()

    idx = indices[search_key]

    # Get pairwise similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Filter out target title itself, slice top_n
    top_matches = sim_scores[1:top_n + 1]
    movie_indices = [i[0] for i in top_matches]
    raw_scores = [i[1] for i in top_matches]

    recs = df.iloc[movie_indices][['show_id', 'title', 'type', 'listed_in', 'director', 'country', 'release_year']].copy()
    recs['similarity_score'] = [f"{round(score * 100, 2)}%" for score in raw_scores]
    recs['score_num'] = raw_scores
    return recs.reset_index(drop=True)

In [7]:
# Test Recommendation Engine on selected benchmark titles
demo_titles = ["Squid Game", "Jaws", "Grown Ups"]

for t in demo_titles:
    print("=" * 70)
    print(f"Recommendations for: '{t}'")
    print("=" * 70)
    results = get_recommendations(t, top_n=5)
    display(results[['title', 'type', 'release_year', 'listed_in', 'similarity_score']])
    print("\n")

Recommendations for: 'Squid Game'


,title,type,release_year,listed_in,similarity_score
0,Kakegurui,TV Show,2019,"International TV Shows, TV Dramas, TV Thrillers",100.0%
1,The Spy,TV Show,2019,"International TV Shows, TV Dramas, TV Thrillers",89.86%
2,Marseille,TV Show,2018,"International TV Shows, TV Dramas, TV Thrillers",89.86%
3,Fatma,TV Show,2021,"International TV Shows, TV Dramas, TV Thrillers",88.39%
4,50M2,TV Show,2021,"International TV Shows, TV Dramas, TV Thrillers",88.39%




Recommendations for: 'Jaws'


,title,type,release_year,listed_in,similarity_score
0,Indiana Jones and the Raiders of the Lost Ark,Movie,1981,"Action & Adventure, Children & Family Movies, ...",88.3%
1,Indiana Jones and the Temple of Doom,Movie,1984,"Action & Adventure, Children & Family Movies, ...",88.3%
2,Schindler's List,Movie,1993,"Classic Movies, Dramas",87.01%
3,Indiana Jones and the Last Crusade,Movie,1989,"Action & Adventure, Children & Family Movies, ...",80.26%
4,Rocky,Movie,1976,"Action & Adventure, Classic Movies, Dramas",70.81%




Recommendations for: 'Grown Ups'


,title,type,release_year,listed_in,similarity_score
0,Big Daddy,Movie,1999,Comedies,100.0%
1,Beverly Hills Ninja,Movie,1997,"Action & Adventure, Comedies",88.37%
2,You Don't Mess with the Zohan,Movie,2008,"Action & Adventure, Comedies",64.29%
3,Bad Trip,Movie,2021,Comedies,44.73%
4,Still LAUGH-IN: The Stars Celebrate,Movie,2019,Comedies,44.73%


## **Step 5: Evaluate Recommendation Quality**

To quantitatively evaluate our content-based recommender, we assess:
1. **Genre Overlap (Jaccard Similarity Index):**
   $$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$
   Measures the fraction of shared genres between target and recommended titles.
2. **Content Type Consistency Rate:**
   Percentage of recommendations that preserve content format (Movie $\to$ Movie, TV Show $\to$ TV Show).
3. **Mean Cosine Similarity:**
   Average confidence score across top recommendations.

In [8]:
def evaluate_recommendations(benchmark_titles=None, top_n=5):
    """
    Computes evaluation metrics across a suite of diverse test titles.
    """
    if benchmark_titles is None:
        benchmark_titles = [
            "Jaws",
            "Squid Game",
            "Grown Ups",
            "Dark Skies",
            "Naruto Shippuden the Movie: Blood Prison"
        ]

    def get_genre_set(genre_str):
        return set(g.strip().lower() for g in genre_str.split(',') if g.strip())

    jaccard_scores = []
    type_matches = []
    mean_similarities = []

    print("=" * 75)
    print("EVALUATING RECOMMENDATION QUALITY ON BENCHMARK TITLES")
    print("=" * 75)

    for title in benchmark_titles:
        recs = get_recommendations(title, top_n=top_n)
        if recs.empty:
            continue

        target_idx = indices[title.lower()]
        target_row = df.iloc[target_idx]
        target_genres = get_genre_set(target_row['listed_in'])
        target_type = target_row['type']

        cur_jaccards = []
        cur_type_matches = []

        for _, row in recs.iterrows():
            rec_genres = get_genre_set(row['listed_in'])
            intersection = len(target_genres.intersection(rec_genres))
            union = len(target_genres.union(rec_genres))
            cur_jaccards.append(intersection / union if union > 0 else 0)
            cur_type_matches.append(1 if row['type'] == target_type else 0)

        avg_j = np.mean(cur_jaccards)
        avg_t = np.mean(cur_type_matches) * 100
        avg_s = np.mean(recs['score_num']) * 100

        jaccard_scores.append(avg_j)
        type_matches.append(avg_t)
        mean_similarities.append(avg_s)

        print(f"Target: [{target_type}] '{target_row['clean_title']}'")
        print(f"  - Mean Genre Overlap (Jaccard) : {avg_j:.2f}")
        print(f"  - Format Match Rate            : {avg_t:.0f}%")
        print(f"  - Mean Similarity Score        : {avg_s:.1f}%\n")

    # Summary Report
    print("=" * 75)
    print("OVERALL QUALITY METRICS SUMMARY")
    print("=" * 75)
    summary_df = pd.DataFrame({
        'Metric': [
            'Mean Genre Overlap (Jaccard Index)',
            'Content Format Consistency Rate',
            'Mean Top-N Similarity Confidence'
        ],
        'Score': [
            f"{np.mean(jaccard_scores):.3f} / 1.000",
            f"{np.mean(type_matches):.1f}%",
            f"{np.mean(mean_similarities):.1f}%"
        ]
    })
    display(summary_df)

evaluate_recommendations()

EVALUATING RECOMMENDATION QUALITY ON BENCHMARK TITLES
Target: [Movie] 'Jaws'
  - Mean Genre Overlap (Jaccard) : 0.63
  - Format Match Rate            : 100%
  - Mean Similarity Score        : 82.9%

Target: [TV Show] 'Squid Game'
  - Mean Genre Overlap (Jaccard) : 1.00
  - Format Match Rate            : 100%
  - Mean Similarity Score        : 91.3%

Target: [Movie] 'Grown Ups'
  - Mean Genre Overlap (Jaccard) : 0.80
  - Format Match Rate            : 100%
  - Mean Similarity Score        : 68.4%

Target: [Movie] 'Dark Skies'
  - Mean Genre Overlap (Jaccard) : 0.67
  - Format Match Rate            : 100%
  - Mean Similarity Score        : 65.3%

Target: [Movie] 'Naruto Shippuden the Movie: Blood Prison'
  - Mean Genre Overlap (Jaccard) : 1.00
  - Format Match Rate            : 100%
  - Mean Similarity Score        : 90.7%

OVERALL QUALITY METRICS SUMMARY


,Metric,Score
0,Mean Genre Overlap (Jaccard Index),0.820 / 1.000
1,Content Format Consistency Rate,100.0%
2,Mean Top-N Similarity Confidence,79.7%


## **Interactive Title Query (Colab Ready)**

Type any movie or TV show title in the field below to get recommendations:

In [10]:
#@title Netflix Recommendation Finder
title_to_search = "Squid Game" #@param {type:"string"}
number_of_recommendations = 5 #@param {type:"slider", min:1, max:10, step:1}

recommendations = get_recommendations(title_to_search, top_n=number_of_recommendations)
if not recommendations.empty:
    print(f"\nTop {number_of_recommendations} Recommendations for '{title_to_search}':\n")
    display(recommendations[['title', 'type', 'release_year', 'listed_in', 'director', 'country', 'similarity_score']])


Top 5 Recommendations for 'Squid Game':



,title,type,release_year,listed_in,director,country,similarity_score
0,Kakegurui,TV Show,2019,"International TV Shows, TV Dramas, TV Thrillers",,Pakistan,100.0%
1,The Spy,TV Show,2019,"International TV Shows, TV Dramas, TV Thrillers",,France,89.86%
2,Marseille,TV Show,2018,"International TV Shows, TV Dramas, TV Thrillers",,France,89.86%
3,Fatma,TV Show,2021,"International TV Shows, TV Dramas, TV Thrillers",,Turkey,88.39%
4,50M2,TV Show,2021,"International TV Shows, TV Dramas, TV Thrillers",,Turkey,88.39%
